# 02 — Transaction Preprocessing & Invoice-Level Sampling
**Group 6 | Topic 6: Market Basket Analysis on Online Retail II | `SEED = 5018`**

This notebook demonstrates and verifies the preprocessing pipeline:
1. Removing cancelled invoices (`'C'` prefix) and invalid quantities/prices
2. Sampling strictly by unique `InvoiceNo` (`random_state=5018`) rather than individual rows
3. Building the boolean transaction-item basket matrix for `mlxtend`

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import GROUP_SEED
from src.download_data import download_dataset
from src.preprocessing import (
    clean_transactions,
    create_basket_matrix,
    load_raw_transactions,
    sample_invoices,
)

SEED = GROUP_SEED  # 5018
N_INVOICES = 5000
np.random.seed(SEED)

## 1. Cancellation Filtering & Transaction Cleaning

In [ ]:
dataset_path = download_dataset(data_dir=PROJECT_ROOT / "data")
raw_df = load_raw_transactions(dataset_path)
cleaned_df = clean_transactions(raw_df, drop_missing_customer_id=True)

print(f"Raw rows: {len(raw_df):,} -> Cleaned rows: {len(cleaned_df):,}")
print(f"Valid unique invoices available: {cleaned_df['InvoiceNo'].nunique():,}")

## 2. Deterministic Invoice-Level Sampling (`random_state=5018`)

In [ ]:
# Sample unique invoices (never individual rows) to preserve complete market baskets
sampled_df = sample_invoices(cleaned_df, seed=SEED, n_invoices=N_INVOICES)

# Verify that every sampled invoice kept 100% of its cleaned product lines
sampled_counts = sampled_df.groupby("InvoiceNo").size()
expected_counts = cleaned_df.loc[
    cleaned_df["InvoiceNo"].isin(sampled_counts.index)
].groupby("InvoiceNo").size()

assert sampled_counts.equals(expected_counts), "Basket integrity check failed!"
print(f"Verified 100% basket integrity across {len(sampled_counts):,} sampled invoices.")

## 3. Sparse Boolean Invoice-by-Item Basket Matrix Construction

In [ ]:
basket_matrix = create_basket_matrix(sampled_df, sparse=True)
print(f"Basket matrix shape (invoices x items): {basket_matrix.shape}")
print(f"Matrix density: {basket_matrix.sparse.density * 100:.4f}%")
print(f"Memory footprint: {basket_matrix.memory_usage(deep=True).sum() / (1024 ** 2):.3f} MB")

basket_matrix.iloc[:5, :8]